# 🌅 SUNDAY AI — Full Training Pipeline

> **GPT-style Language Model — Trained from Scratch on Google Colab T4 GPU**

---

## 📋 What This Notebook Does

| Step | Description |
|------|-------------|
| 1️⃣ | Setup: Mount Drive, clone repo, install deps |
| 2️⃣ | Verify GPU (T4 required) |
| 3️⃣ | Train BPE Tokenizer (32K vocab) |
| 4️⃣ | Download & Prepare Training Data |
| 5️⃣ | Initialize SUNDAY Model |
| 6️⃣ | Pretrain on OpenWebText |
| 7️⃣ | Evaluate (Perplexity + Generation) |
| 8️⃣ | Fine-tune on Alpaca (instruction following) |
| 9️⃣ | Save to Google Drive |

---

**⚡ Runtime: Set to GPU → T4**  
Runtime → Change runtime type → Hardware accelerator → **T4 GPU**

## 🔧 Step 1: Environment Setup

In [ ]:
# --- 1a. Mount Google Drive (for saving checkpoints) ---
from google.colab import drive
drive.mount('/content/drive')

# Create a folder in Drive for SUNDAY checkpoints
import os
DRIVE_DIR = '/content/drive/MyDrive/SUNDAY_AI'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Checkpoints will be saved to: {DRIVE_DIR}')

In [ ]:
# --- 1b. Clone SUNDAY AI Repository ---
# IMPORTANT: Replace with YOUR GitHub username!
GITHUB_USERNAME = 'YOUR_USERNAME'  # ← Change this!
REPO_NAME = 'SUNDAY_AI'

%cd /content

# If already cloned, just pull latest
if os.path.exists(f'/content/{REPO_NAME}'):
    %cd /content/{REPO_NAME}
    !git pull
else:
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
    %cd /content/{REPO_NAME}

print(f'✅ Repository ready at /content/{REPO_NAME}')
!ls -la

In [ ]:
# --- 1c. Install Dependencies ---
# Note: PyTorch is pre-installed in Colab with CUDA support
!pip install -r requirements-colab.txt -q

# Add project root to Python path
import sys
sys.path.insert(0, '/content/SUNDAY_AI')

print('✅ Dependencies installed')

## ⚡ Step 2: Verify GPU

In [ ]:
import torch

print('=' * 50)
print(f'PyTorch version:  {torch.__version__}')
print(f'CUDA available:   {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:              {gpu_name}')
    print(f'GPU Memory:       {gpu_mem:.1f} GB')
    DEVICE = 'cuda'
    print('✅ GPU is ready for training!')
else:
    print('⚠️  No GPU found. Go to: Runtime → Change runtime type → GPU → T4')
    DEVICE = 'cpu'

print('=' * 50)

## 🔤 Step 3: Train BPE Tokenizer

We train a 32K vocabulary BPE tokenizer on a sample of our training data.  
**Skip this step if you already have a tokenizer** in `tokenizer_files/`.

In [ ]:
import os

TOKENIZER_PATH = 'tokenizer_files/sunday_tokenizer.json'

if os.path.exists(TOKENIZER_PATH):
    print(f'✅ Tokenizer already exists at {TOKENIZER_PATH}. Skipping training.')
    # Load it
    from src.tokenizer.tokenizer import SundayTokenizer
    tokenizer = SundayTokenizer.from_file(TOKENIZER_PATH)
    print(f'   Vocab size: {tokenizer.vocab_size:,}')
else:
    print('🔤 Training BPE tokenizer from scratch...')
    
    # Quick approach: train on a small OpenWebText subset
    from datasets import load_dataset
    
    print('Downloading 50K samples from OpenWebText for tokenizer training...')
    dataset = load_dataset('openwebtext', split='train', streaming=True)
    
    # Collect 50K texts for tokenizer training
    texts = []
    for i, example in enumerate(dataset):
        texts.append(example['text'])
        if i >= 49999:
            break
    
    print(f'Collected {len(texts):,} texts for tokenizer training')
    
    # Save to temp file for trainer
    os.makedirs('data', exist_ok=True)
    with open('data/tokenizer_training_text.txt', 'w', encoding='utf-8') as f:
        for t in texts:
            f.write(t.replace('\n', ' ') + '\n')
    
    # Train tokenizer
    from src.tokenizer.trainer import train_tokenizer
    
    tokenizer_hf = train_tokenizer(
        data_files=['data/tokenizer_training_text.txt'],
        vocab_size=32768,
        min_frequency=2,
        save_dir='tokenizer_files',
        save_name='sunday_tokenizer.json',
    )
    
    # Load wrapper
    from src.tokenizer.tokenizer import SundayTokenizer
    tokenizer = SundayTokenizer.from_file(TOKENIZER_PATH)
    print(f'✅ Tokenizer trained! Vocab size: {tokenizer.vocab_size:,}')
    
    # Copy to Drive
    import shutil
    shutil.copy(TOKENIZER_PATH, f'{DRIVE_DIR}/sunday_tokenizer.json')
    print(f'✅ Tokenizer saved to Google Drive')

In [ ]:
# Quick tokenizer test
test_text = 'The transformer model learns to predict the next token."
tokens = tokenizer.get_tokens(test_text)
ids = tokenizer.encode(test_text)
decoded = tokenizer.decode(ids)

print(f'Text:    {test_text}')
print(f'Tokens:  {tokens}')
print(f'IDs:     {ids[:10]}...')
print(f'Decoded: {decoded}')
assert decoded == test_text, 'Encode/decode mismatch!'
print('✅ Tokenizer works correctly!')

## 📦 Step 4: Prepare Training Data

We download OpenWebText and tokenize it into binary files for fast training.  
**This takes ~20-40 minutes** for the full dataset.

> 💡 **Tip:** For a quick test run, set `MAX_SAMPLES = 50_000` (takes ~5 min).

In [ ]:
# --- Configuration ---
MAX_SAMPLES = 500_000   # Set to None for the full ~8M doc dataset
                         # 100K → ~15 min prep, good for quick experiments
                         # 500K → ~1 hour prep, decent pretraining
TRAIN_BIN = 'data/train.bin'
VAL_BIN   = 'data/val.bin'

if os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN):
    import numpy as np
    train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode='r')
    val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode='r')
    print(f'✅ Binary data files already exist!')
    print(f'   Train: {len(train_data):,} tokens')
    print(f'   Val:   {len(val_data):,} tokens')
else:
    print(f'📦 Downloading and tokenizing {MAX_SAMPLES or "all"} OpenWebText docs...')
    print('   (This may take 20-60 minutes depending on dataset size)')
    
    from datasets import load_dataset
    from src.data.preprocessing import prepare_text_corpus
    
    # Download OpenWebText
    dataset = load_dataset('openwebtext', split='train', streaming=MAX_SAMPLES is not None)
    
    if MAX_SAMPLES:
        texts = []
        for i, ex in enumerate(dataset):
            texts.append(ex['text'])
            if i >= MAX_SAMPLES - 1:
                break
    else:
        texts = dataset['text']
    
    print(f'   Downloaded {len(texts):,} documents')
    
    # Tokenize and save as binary
    train_path, val_path = prepare_text_corpus(
        texts=texts,
        tokenizer=tokenizer,
        output_dir='data',
        train_split=0.95,
        verbose=True,
    )
    
    print(f'\n✅ Data preparation complete!')
    print(f'   Saved to: {train_path}, {val_path}')

In [ ]:
# --- Create PyTorch Datasets & DataLoaders ---
from src.data.dataset import SundayDatasetFromFile
from src.data.dataloader import create_dataloader_from_datasets

# Model config (choose your size!)
MODEL_SIZE = 'tiny'   # Start here for testing
# MODEL_SIZE = '50m'  # ~6-8 GB VRAM, good for T4
# MODEL_SIZE = '125m' # ~12-15 GB VRAM, needs careful memory management

from configs.model_config import get_model_config
model_config = get_model_config(MODEL_SIZE)
MAX_SEQ_LEN = model_config['max_seq_len']

# Batch sizes (adjust based on your VRAM)
BATCH_SIZE_MAP = {
    'tiny':  32,   # Very small model, large batch OK
    '50m':    4,   # Adjust based on VRAM (try 4-8)
    '125m':   2,   # Tight on T4, use grad accumulation
}
BATCH_SIZE = BATCH_SIZE_MAP[MODEL_SIZE]
GRAD_ACCUM = 8 if MODEL_SIZE in ['50m', '125m'] else 1  # Effective batch = BATCH × ACCUM

# Create datasets
train_dataset = SundayDatasetFromFile(TRAIN_BIN, max_seq_len=MAX_SEQ_LEN)
val_dataset   = SundayDatasetFromFile(VAL_BIN,   max_seq_len=MAX_SEQ_LEN)

# Create DataLoaders
# num_workers=2 on Colab Linux (use 0 on Windows)
train_loader, val_loader = create_dataloader_from_datasets(
    train_dataset, val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True,  # Faster CPU→GPU transfer
)

print(f'\n✅ DataLoaders ready!')
print(f'   Model size:     {MODEL_SIZE}')
print(f'   Batch size:     {BATCH_SIZE}')
print(f'   Grad accum:     {GRAD_ACCUM}')
print(f'   Effective batch: {BATCH_SIZE * GRAD_ACCUM}')

## 🧠 Step 5: Initialize Model

In [ ]:
from src.model.sunday_model import SundayModel
import torch

device = torch.device(DEVICE)

# Initialize model
model = SundayModel.from_config(MODEL_SIZE)
model.print_info()
model.to(device)

# Check VRAM usage
if DEVICE == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\nVRAM used by model: {allocated:.2f} GB / {total:.1f} GB')

## 🏋️ Step 6: Pretrain the Model

**Estimated training time (T4 GPU):**

| Model | Steps | Batch | Time |
|-------|-------|-------|------|
| Tiny (5M) | 10K | 32 | ~15 min |
| 50M | 50K | 4×8 | ~4-6 hours |
| 125M | 100K | 2×16 | ~10-15 hours |

> ⚠️ **Colab disconnects after ~12 hours.** The trainer auto-saves checkpoints and resumes automatically.

In [ ]:
# --- (Optional) W&B Setup ---
# Uncomment to track your training in Weights & Biases
# import wandb
# wandb.login()  # You'll be prompted for your API key

In [ ]:
from src.training.trainer import Trainer

# --- Training Configuration ---
TRAINING_CONFIG = {
    # Learning Rate
    'max_lr': {
        'tiny': 3e-3,
        '50m':  6e-4,
        '125m': 3e-4,
    }[MODEL_SIZE],
    'min_lr': {
        'tiny': 3e-4,
        '50m':  6e-5,
        '125m': 3e-5,
    }[MODEL_SIZE],
    'warmup_steps': {
        'tiny': 200,
        '50m':  2000,
        '125m': 2000,
    }[MODEL_SIZE],
    
    # Training Duration
    'max_steps': {
        'tiny': 10_000,
        '50m':  50_000,
        '125m': 100_000,
    }[MODEL_SIZE],
    
    # Batch & Accumulation
    'batch_size': BATCH_SIZE,
    'grad_accum_steps': GRAD_ACCUM,
    
    # Regularization
    'weight_decay': 0.1,
    'beta1': 0.9,
    'beta2': 0.95,
    'grad_clip': 1.0,
    
    # Mixed Precision (auto-enabled on CUDA)
    'use_amp': True,
    
    # Logging & Evaluation
    'log_interval': 50,
    'val_interval': 500,
    'save_interval': 1000,
    
    # Checkpointing (save to Drive!)
    'save_dir': f'{DRIVE_DIR}/checkpoints',
    
    # W&B (set to True after wandb.login())
    'wandb_enabled': False,
    'wandb_project': 'SUNDAY_AI',
}

# --- Initialize Trainer ---
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=TRAINING_CONFIG,
    device=DEVICE,
)

print('✅ Trainer initialized. Ready to train!')
print(f'   Checkpoints will auto-save to: {DRIVE_DIR}/checkpoints')

In [ ]:
# 🚀 START TRAINING!
# If Colab disconnects, re-run from Step 1 — it will auto-resume from last checkpoint.

print('🚀 Starting SUNDAY pretraining...')
print('Press the Stop button to interrupt. Progress is saved every 1000 steps.')
print('=' * 60)

trainer.train()

## 📊 Step 7: Evaluate the Model

In [ ]:
from src.evaluation.evaluator import Evaluator

evaluator = Evaluator(model, tokenizer, device)

# Sample prompts to test generation quality
sample_prompts = [
    'The transformer architecture was introduced in",
    'Once upon a time in a land far away,',
    'Machine learning is the field of study that',
    'The capital of France is',
    'To train a neural network, you need to',
]

results = evaluator.evaluate(
    val_loader=val_loader,
    prompts=sample_prompts,
    max_batches=50,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
)

print(f'\n📊 EVALUATION RESULTS:')
print(f'   Perplexity: {results["perplexity"]:.2f}')
if 'generation_metrics' in results:
    gm = results['generation_metrics']
    print(f'   Distinct-1: {gm["distinct_1"]:.3f}')
    print(f'   Distinct-2: {gm["distinct_2"]:.3f}')

# Print samples
if 'samples' in results:
    print('\n📝 GENERATION SAMPLES:')
    print('=' * 60)
    for prompt, continuation in results['samples']:
        print(f'Prompt: {prompt}')
        print(f'Output: {continuation[:200]}')
        print('-' * 60)

## 💡 Step 8: Fine-tune on Alpaca (Instruction Following)

After pretraining, we fine-tune to teach the model to follow instructions.  
**Run this AFTER pretraining completes.**

In [ ]:
from src.finetuning.data_formatter import prepare_sft_dataset, AlpacaDataset
from src.finetuning.sft_trainer import SFTTrainer
from src.data.download import download_alpaca
from torch.utils.data import DataLoader, random_split

print('📥 Downloading Stanford Alpaca dataset...')
alpaca_examples = download_alpaca()

print('\n🔄 Preparing SFT dataset (tokenizing + formatting)...')
prepared_data = prepare_sft_dataset(
    examples=alpaca_examples,
    tokenizer=tokenizer,
    max_seq_len=MAX_SEQ_LEN,
)

# Split into train/val
full_dataset = AlpacaDataset(prepared_data, max_seq_len=MAX_SEQ_LEN)
n_train = int(len(full_dataset) * 0.95)
n_val = len(full_dataset) - n_train
sft_train, sft_val = random_split(full_dataset, [n_train, n_val])

sft_train_loader = DataLoader(sft_train, batch_size=2, shuffle=True, num_workers=2)
sft_val_loader   = DataLoader(sft_val,   batch_size=2, shuffle=False, num_workers=2)

print(f'\n✅ Alpaca SFT dataset ready!')
print(f'   Train: {n_train:,} examples')
print(f'   Val:   {n_val:,} examples')

In [ ]:
from configs.training_config import FINETUNING_CONFIG

# Update checkpoint dir to Drive
ft_config = dict(FINETUNING_CONFIG)
ft_config['checkpoint_dir'] = f'{DRIVE_DIR}/checkpoints/finetuned'
ft_config['max_epochs'] = 3

sft_trainer = SFTTrainer(
    model=model,
    train_loader=sft_train_loader,
    val_loader=sft_val_loader,
    config=ft_config,
    device=DEVICE,
)

print('💡 Starting Alpaca fine-tuning...')
sft_trainer.train()

## 💬 Step 9: Chat with SUNDAY!

In [ ]:
from src.inference.generator import Generator
from src.inference.chat import ChatSession

generator = Generator(model, tokenizer, device)
session = ChatSession(generator, instruction_tuned=True)

# Interactive chat loop
print('💬 Chat with SUNDAY AI! (type "quit" to exit, "reset" to clear history)')
print('=' * 60)

while True:
    user_input = input('You: ').strip()
    if not user_input:
        continue
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
    if user_input.lower() == 'reset':
        session.reset()
        continue
    
    response = session.chat(
        user_input,
        max_new_tokens=200,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
    )
    print(f'SUNDAY: {response}\n')

## 💾 Step 10: Save Everything to Drive

In [ ]:
import shutil

print('💾 Saving all artifacts to Google Drive...')

# 1. Save tokenizer
shutil.copy('tokenizer_files/sunday_tokenizer.json', f'{DRIVE_DIR}/sunday_tokenizer.json')
print(f'✅ Tokenizer saved')

# 2. Save final model
import torch
torch.save({
    'model_state_dict': model.state_dict(),
    'config': model.config,
}, f'{DRIVE_DIR}/sunday_final_model.pt')
print(f'✅ Final model saved')

# 3. Save model info
params = model.count_parameters()
info = f"""
SUNDAY AI - Training Summary
============================
Model size: {MODEL_SIZE}
Total params: {params['total']:,}
Device: {DEVICE}
Steps trained: {trainer.step:,}
Best val loss: {trainer.best_val_loss:.4f}
"""
with open(f'{DRIVE_DIR}/training_summary.txt', 'w') as f:
    f.write(info)
print(f'✅ Training summary saved')
print(info)
print(f'\n📁 All files saved to: {DRIVE_DIR}')